
This project analyzes Billboard chart data to measure market concentration in the music industry.

We focus on:
- Data cleaning
- Artist frequency distribution
- Herfindahl-Hirschman Index (HHI)
- Market concentration trends

In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect("../all_data/data.db")
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,BillboardCharts
1,IndividualArtistMentions
2,SongArtistMentions
3,SongArtistChartAppearances
4,SpotifyDailyStreamingCharts
5,SpotifyWeeklyStreamingCharts
6,AppleChartRankings


In [2]:
df = pd.read_sql_query("SELECT * FROM BillboardCharts LIMIT 5;", conn)
df

,artist,song_name,rank,artist_link,last_week_rank,peak,weeks_chart,week,chart
0,Flipperachi,FA9LA,1,None,1,1,10,02-28-2026,Billboard Arabic Hot 100
1,DYSTINCT,Yama,2,None,2,1,17,02-28-2026,Billboard Arabic Hot 100
2,Vanco Featuring AYA,Ma Tnsani,3,None,3,2,29,02-28-2026,Billboard Arabic Hot 100
3,"Lege-Cy, Ghaliaa",Msh Awl Marra,4,None,7,4,3,02-28-2026,Billboard Arabic Hot 100
4,DYSTINCT,Ta3al,5,None,4,3,4,02-28-2026,Billboard Arabic Hot 100


In [3]:
df.columns

Index(['artist', 'song_name', 'rank', 'artist_link', 'last_week_rank', 'peak',
       'weeks_chart', 'week', 'chart'],
      dtype='object')


We define the market as the Billboard Arabic Hot 100 chart.
Each week consists of ranked songs, and artists compete for chart presence.

We measure market concentration using artist frequency within the chart.

In [4]:
df.shape

(5, 9)

In [5]:
df['week'].nunique()

1

In [6]:
df = pd.read_sql_query("SELECT * FROM BillboardCharts;", conn)

df.shape

(1650, 9)

In [7]:
df['week'].nunique()

2

In [8]:
sorted(df['week'].unique())[:5]

['02-21-2026', '02-28-2026']

In [9]:
df['chart'].unique()

array(['Billboard Arabic Hot 100', 'Billboard Argentina Hot 100',
       'Billboard Brasil Hot 100', 'Billboard Colombia Hot 100',
       'Billboard Canadian Hot 100', 'Billboard Italy Hot 100',
       'Billboard Japan Hot 100', 'Billboard Korea Hot 100',
       'Billboard Philippines Hot 100',
       'Billboard Thailand Top Thai Songs', 'Billboard Vietnam Hot 100',
       'Australia Songs', 'Austria Songs', 'Belgium Songs',
       'Bolivia Songs', 'Chile Songs', 'China TME UNI Chart',
       'Croatia Songs', 'Czech Republic Songs', 'Denmark Songs',
       'Ecuador Songs', 'Finland Songs', 'France Songs', 'Germany Songs',
       'Greece Songs', 'Hong Kong Songs', 'Hungary Songs',
       'Iceland Songs', 'India Songs', 'Indonesia Songs', 'Ireland Songs',
       'Luxembourg Songs', 'Malaysia Songs', 'Mexico Songs',
       'Netherlands Songs', 'New Zealand Songs', 'Norway Songs',
       'Peru Songs', 'Poland Songs', 'Portugal Songs', 'Romania Songs',
       'Singapore Songs', 'Slovakia So

Each country-specific Billboard chart is treated as an independent music market.

We calculate the Herfindahl-Hirschman Index (HHI) for each country to measure artist concentration within that market.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df['week'] = pd.to_datetime(df['week'])

In [11]:
hhi_country = df.groupby('chart')['artist'] \
    .apply(lambda x: (x.value_counts(normalize=True) ** 2).sum())

hhi_country = hhi_country.sort_values(ascending=False)

hhi_country.head(10)

chart
Billboard Korea Hot 100    0.3952
Iceland Songs              0.2384
Hong Kong Songs            0.1616
Croatia Songs              0.1520
Ecuador Songs              0.1392
Peru Songs                 0.1360
Bolivia Songs              0.1328
Chile Songs                0.1296
New Zealand Songs          0.1168
Mexico Songs               0.1104
Name: artist, dtype: float64

1. Compute Country-Level Artist Market Shares

For each country chart, we compute artist market shares based on chart appearances.
Market share is defined as the proportion of songs attributed to each artist within a given country.

In [12]:
# artist market share
country_artist_share = df.groupby(['chart', 'artist']) \
    .size() \
    .reset_index(name='count')

total_songs_country = country_artist_share.groupby('chart')['count'].transform('sum')

country_artist_share['market_share'] = country_artist_share['count'] / total_songs_country

country_artist_share.head()

,chart,artist,count,market_share
0,Australia Songs,Alex Warren,1,0.04
1,Australia Songs,Bad Bunny,1,0.04
2,Australia Songs,Bruno Mars,1,0.04
3,Australia Songs,"Dave, Tems",1,0.04
4,Australia Songs,Djo,1,0.04


2. Herfindahl-Hirschman Index (HHI)

HHI measures market concentration as the sum of squared market shares.
Higher values indicate greater dominance by a few artists.

In [18]:
hhi_country = country_artist_share.groupby('chart')['market_share'] \
    .apply(lambda x: (x ** 2).sum()) \
    .reset_index(name='HHI')

hhi_country.head()

hhi_country.to_csv("hhi_country.csv", index=False)

3. Concentration Ratio (CR3)

CR3 measures the combined market share of the top 3 artists in each country.

In [19]:
cr3_country = country_artist_share.sort_values(
    ['chart', 'market_share'],
    ascending=[True, False]
).groupby('chart').head(3)

cr3_country = cr3_country.groupby('chart')['market_share'] \
    .sum() \
    .reset_index(name='CR3')

cr3_country.head()

cr3_country.to_csv("cr3_country.csv", index=False)

4. Gini Coefficient

The Gini coefficient measures inequality in artist market shares.
Values closer to 1 indicate greater inequality.

In [20]:
import numpy as np

def gini(array):
    array = np.sort(array)
    n = len(array)
    cumulative = np.cumsum(array)
    return 1 - 2 * np.sum(cumulative) / (n * cumulative[-1]) + 1/n

gini_country = country_artist_share.groupby('chart')['market_share'] \
    .apply(gini) \
    .reset_index(name='Gini')

gini_country.head()

gini_country.to_csv("gini_country.csv", index=False)

5. Summary of Concentration Measures

In [21]:
summary_table = hhi_country.merge(cr3_country, on='chart') \
    .merge(gini_country, on='chart')

summary_table = summary_table.sort_values('HHI', ascending=False)

summary_table.head(10)

summary_table.to_excel("concentration_results.xlsx", index=False)

In [17]:
song_concentration = df.groupby(['chart', 'song_name']) \
    .size() \
    .reset_index(name='appearances')

song_summary = song_concentration.groupby('chart')['appearances'] \
    .agg(['mean', 'std', 'max']) \
    .reset_index()

song_summary.head()

,chart,mean,std,max
0,Australia Songs,1.0,0.0,1
1,Austria Songs,1.0,0.0,1
2,Belgium Songs,1.0,0.0,1
3,Billboard Arabic Hot 100,1.0,0.0,1
4,Billboard Argentina Hot 100,1.0,0.0,1
